In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Specifying frequency range

minT = 1 # 1second
maxT = 30 # 30 seconds

samples = np.random.default_rng().normal(2)
print(samples)

##### Stuff

In [ ]:
# # Top Floor Read
# data_top = pd.read_csv('top_floor_cleaned.csv')
# data_x3 = data_top.iloc[:, 2].to_numpy()
# data_y3 = data_top.iloc[:, 3].to_numpy()
# data_z3 = data_top.iloc[:, 4].to_numpy()

# # Second Floor Read
# data_2 = pd.read_csv('2nd_floor_cleaned.csv')
# data_x2 = data_2.iloc[:, 2].to_numpy()
# data_y2 = data_2.iloc[:, 3].to_numpy()
# data_z2 = data_2.iloc[:, 4].to_numpy()

# # First Floor Read
# data_1 = pd.read_csv('1st_floor_cleaned.csv')
# data_x1 = data_1.iloc[:, 2].to_numpy()
# data_y1 = data_1.iloc[:, 3].to_numpy()
# data_z1 = data_1.iloc[:, 4].to_numpy()

# # Ground Floor Read
# data_0 = pd.read_csv('ground_floor_cleaned.csv')
# data_x0 = data_0.iloc[:, 2].to_numpy()
# data_y0 = data_0.iloc[:, 3].to_numpy()
# data_z0 = data_0.iloc[:, 4].to_numpy()






data_top = pd.read_csv('top_floor_cleaned.csv')
data_x3 = data_top.iloc[:, 2].to_numpy()
data_y3 = data_top.iloc[:, 3].to_numpy()
data_z3 = data_top.iloc[:, 4].to_numpy()

# Second Floor Read
data_2 = pd.read_csv('2nd_floor_cleaned.csv')
data_x2 = data_2.iloc[:, 2].to_numpy()
data_y2 = data_2.iloc[:, 3].to_numpy()
data_z2 = data_2.iloc[:, 4].to_numpy()

# First Floor Read
data_1 = pd.read_csv('1st_floor_cleaned.csv')
data_x1 = data_1.iloc[:, 2].to_numpy()
data_y1 = data_1.iloc[:, 3].to_numpy()
data_z1 = data_1.iloc[:, 4].to_numpy()

# Ground Floor Read
data_0 = pd.read_csv('ground_floor_cleaned.csv')
data_x0 = data_0.iloc[:, 2].to_numpy()
data_y0 = data_0.iloc[:, 3].to_numpy()
data_z0 = data_0.iloc[:, 4].to_numpy()

Fs = 200  # 200 Hz - data sampling rate
Ts = 1 / Fs  # time interval

# Time columns (Note: t3 is preserved separately for the Power Spectrum calculation)
t3 = np.arange(0, len(data_x3)) * Ts
t2 = np.arange(0, len(data_x2)) * Ts
t1 = np.arange(0, len(data_x1)) * Ts
t0 = np.arange(0, len(data_x0)) * Ts

npts3 = len(data_x3)  # Top Floor
npts2 = len(data_x2)  # 2nd Floor
npts1 = len(data_x1)  # 1st Floor
npts0 = len(data_x0)  # Ground Floor


# Helper for MATLAB's 2^nextpow2(n)
def next_pow2(n):
    return int(2 ** np.ceil(np.log2(n)))


n3 = next_pow2(npts3)
n2 = next_pow2(npts2)
n1 = next_pow2(npts1)
n0 = next_pow2(npts0)

# FFT (Normalized with Fs to compare with SeismoSignal results)
FFT_data_x3 = np.fft.fft(data_x3, n3) / Fs
FFT_data_x2 = np.fft.fft(data_x2, n2) / Fs
FFT_data_x1 = np.fft.fft(data_x1, n1) / Fs
FFT_data_x0 = np.fft.fft(data_x0, n0) / Fs

freq3 = (Fs / 2) * np.linspace(0, 1, n3 // 2 + 1)
freq2 = (Fs / 2) * np.linspace(0, 1, n2 // 2 + 1)
freq1 = (Fs / 2) * np.linspace(0, 1, n1 // 2 + 1)
freq0 = (Fs / 2) * np.linspace(0, 1, n0 // 2 + 1)

FFT_data_x3 = np.abs(FFT_data_x3[: len(freq3)])
FFT_data_x2 = np.abs(FFT_data_x2[: len(freq2)])
FFT_data_x1 = np.abs(FFT_data_x1[: len(freq1)])
FFT_data_x0 = np.abs(FFT_data_x0[: len(freq0)])

# Figure 1: Fourier Amplitude Spectra
plt.figure(1)
plt.plot(freq3, FFT_data_x3, 'b', label='Top Floor')
plt.plot(freq2, FFT_data_x2, 'r', label='2nd Floor')
plt.plot(freq1, FFT_data_x1, 'y', label='1st Floor')
plt.plot(freq0, FFT_data_x0, 'g', label='Ground Floor')
plt.title('Fourier Amplitude Spectra')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Fourier Amplitude')
plt.legend()
plt.grid(True)

# Export FFT_x to Excel
df_fft = pd.DataFrame({'Frequency_Hz': freq3, 'FFT_Amplitude': FFT_data_x3})
df_fft.to_excel('FFT_x.xlsx', sheet_name='Sheet1', index=False, header=False)

# Power Spectrum - SeismoSignal
Arms_x = ((1 / np.max(t3)) * np.trapezoid(data_x3**2, t3)) ** 0.5
psd3 = (1 / (np.pi * (Arms_x**2) * np.max(t3))) * (FFT_data_x3**2)

# Figure 2: Power Spectra
plt.figure(2)
plt.plot(freq3, psd3)
plt.title('Power Spectra')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power')
plt.grid(True)

# Export PSD_x to Sheet3 of Excel
df_psd = pd.DataFrame({'Frequency_Hz': freq3, 'PSD': psd3})
with pd.ExcelWriter('PSD_x.xlsx', engine='openpyxl') as writer:
    # Create empty placeholder sheets to match MATLAB's sheet index 3
    pd.DataFrame().to_excel(writer, sheet_name='Sheet1', index=False)
    pd.DataFrame().to_excel(writer, sheet_name='Sheet2', index=False)
    df_psd.to_excel(writer, sheet_name='Sheet3', index=False, header=False)

plt.show()
